# Examen Parcial 2 — Programación 1 (F12)
## Tormentas Geomagnéticas con la API de NASA / DONKI

**Nombre:** Rubén de Jesus Samayoa Girón
**Carnet:** 202607157
**Fecha:** 21/04/2026
**Punteo total:** 100 puntos

---

| Ejercicio | Tema | Puntos |
|---|---|:---:|
| 1a | Construir la URL y realizar la solicitud HTTP | 10 |
| 1b | Explorar la lista de tormentas con un ciclo `for` | 10 |
| 2 | Calcular el Kp máximo e identificar la tormenta más intensa | 20 |
| 3 | Función `clasificar_kp()` con condicionales | 25 |
| 4 | Función `analizar_tormenta()` que retorna un diccionario | 35 |
| | **Total** | **100** |

---

> **Instrucciones generales**
> - Completa cada ejercicio en la celda indicada.
> - Ejecuta las celdas **en orden** de arriba hacia abajo.
> - Puedes consultar tus apuntes y el notebook de clase `XML_JSON_APIs.ipynb`.
> - Entrega el notebook con **todas las celdas ejecutadas** (con output visible).

---
## Contexto: ¿Qué es una tormenta geomagnética?

El Sol emite constantemente partículas cargadas (viento solar). Cuando ocurre una erupción solar intensa, una ola de partículas choca con el campo magnético de la Tierra y provoca una **tormenta geomagnética**.

Estos eventos pueden:
- Generar auroras boreales visibles en latitudes bajas
- Interferir con satélites GPS y comunicaciones de radio
- En casos extremos, dañar redes eléctricas (como el apagón de Quebec en 1989)

### Índice Kp — escala de intensidad

El **índice Kp** (0–9) mide la perturbación del campo magnético terrestre:

| Kp | Categoría | Descripción |
|---|---|---|
| 0 – 3 | Quieto | Sin tormenta |
| 4 | Activo | Perturbación menor |
| 5 | **G1** — Menor | Auroras en latitudes altas |
| 6 | **G2** — Moderada | Auroras hasta latitud 55° |
| 7 | **G3** — Fuerte | Auroras hasta latitud 50° |
| 8 | **G4** — Severa | Problemas en redes eléctricas |
| 9 | **G5** — Extrema | Apagones posibles, auroras tropicales |

La tormenta de mayo 2024 alcanzó **G5** — la más intensa en 20 años.

---
## El endpoint: DONKI / GST

**DONKI** = Space Weather Database Of Notifications, Knowledge, Information  
**GST** = Geomagnetic Storm

```
GET https://api.nasa.gov/DONKI/GST
```

| Parámetro | Tipo | Default | Descripción |
|---|---|---|---|
| `startDate` | YYYY-MM-DD | 30 días atrás | Inicio del rango de fechas |
| `endDate` | YYYY-MM-DD | hoy | Fin del rango de fechas |
| `api_key` | string | DEMO_KEY | Tu API key de api.nasa.gov |

**Ejemplo de URL:**
```
https://api.nasa.gov/DONKI/GST?startDate=2024-05-01&endDate=2024-05-31&api_key=DEMO_KEY|
```

### Estructura de la respuesta

La API devuelve una **lista** de eventos. Cada evento es un diccionario con esta estructura:

```json
{
  "gstID":    "2024-05-10T17:00:00-GST-001",
  "startTime": "2024-05-10T17:00Z",
  "allKpIndex": [
    { "observedTime": "2024-05-10T21:00Z", "kpIndex": 8.67, "source": "NOAA" },
    { "observedTime": "2024-05-11T00:00Z", "kpIndex": 9.0,  "source": "NOAA" }
  ],
  "linkedEvents": [
    { "activityID": "2024-05-08T21:08:00-CME-001" }
  ],
  "link": "https://webtools.ccmc.gsfc.nasa.gov/DONKI/view/GST/..."
}
```

**Campos importantes:**
- `gstID` — identificador único del evento
- `startTime` — cuándo comenzó la tormenta (formato ISO 8601 UTC)
- `allKpIndex` — lista de mediciones Kp durante la tormenta
- `linkedEvents` — eventos espaciales relacionados (erupciones, CME) — puede ser `None`

---
## Paso 1 — Configuración de la API Key

### Obtener tu API key (gratis)

1. Ve a **https://api.nasa.gov**
2. Completa el formulario con tu nombre y correo electrónico
3. Haz clic en **"Signup"**
4. Revisa tu correo — recibirás la key en minutos
5. La key tiene este formato: `TU_API_KEY_AQUI_XXXXXXXXXXXXXXXXXXXXXXXX`

Con tu propia key puedes hacer **1,000 solicitudes por hora** (vs 30 con DEMO_KEY).

---

### Configurar la variable de entorno `NASA_API_KEY`

Una variable de entorno guarda tu key **fuera del código**, para que no quede expuesta  
en el archivo del notebook ni en el historial de git.

#### Linux / macOS — temporal (solo la sesión actual)
```bash
export NASA_API_KEY="tu_key_aqui"
jupyter notebook
```

#### Linux / macOS — permanente
Agrega esta línea al final de `~/.bashrc` (bash) o `~/.zshrc` (zsh):
```bash
echo 'export NASA_API_KEY="tu_key_aqui"' >> ~/.bashrc
source ~/.bashrc   # aplicar sin reiniciar
```

#### Windows — PowerShell (temporal)
```powershell
$env:NASA_API_KEY = "tu_key_aqui"
jupyter notebook
```

#### Windows — permanente (Panel de control)
1. Busca **"Variables de entorno"** en el menú Inicio
2. Clic en **"Editar las variables de entorno del sistema"**
3. Botón **"Variables de entorno..."**
4. En "Variables de usuario" → **Nueva**
5. Nombre: `NASA_API_KEY` | Valor: tu key
6. Aceptar y **reiniciar** Jupyter

#### Verificar que está configurada
```bash
# Linux/macOS
echo $NASA_API_KEY

# Windows PowerShell
echo $env:NASA_API_KEY
```

In [13]:
import os
import requests
import json
import time

# Cargar la API key desde la variable de entorno.-------------------------------------
API_KEY = os.getenv("NASA_APIKEY", "").strip() or "DEMO_KEY"

if API_KEY == "DEMO_KEY":
    print("⚠  Usando DEMO_KEY — límite: 30 solicitudes/hora")
    print("   Configura NASA_APIKEY para mayor límite (ver instrucciones arriba).")
else:
    print(f"✓  API key cargada: {API_KEY[:4]}{'*' * (len(API_KEY) - 4)}")

print("\nLibrerías importadas correctamente ✓")

✓  API key cargada: b6ri************************************

Librerías importadas correctamente ✓


---
## Identificacion del estudiante

Completa tu nombre y carnet en la siguiente celda y **ejecutala antes de comenzar**.  
Estos datos quedan registrados en el output del notebook junto con la posicion de la ISS  
en el momento exacto en que realizas el examen.

In [14]:
NOMBRE = "Rubén de Jesús Samayoa Girón"   
CARNET = "202607157"         

if NOMBRE == "Nombre Apellido" or CARNET == "202300000":
    raise ValueError("Completa tu nombre y carnet antes de continuar.")

print(f"Estudiante : {NOMBRE}")
print(f"Carnet     : {CARNET    }")

Estudiante : Rubén de Jesús Samayoa Girón
Carnet     : 202607157


---
## Ejercicio 1 — Obtener y explorar los datos (20 pts)

### Parte a) — Solicitud al endpoint (10 pts)

Se te da la URL base del endpoint. Construye la URL completa usando un f-string e incluye los parámetros `startDate`, `endDate` y `api_key`. Luego realiza la solicitud y verifica que fue exitosa.

---

### ¿Por qué verificar el código de estado antes de leer el JSON?

Cuando hacemos una solicitud HTTP el servidor siempre responde con un **código de estado numérico** que indica si todo salió bien o hubo un problema.

El código **200** significa *"OK — la solicitud fue exitosa y el cuerpo de la respuesta contiene los datos pedidos"*.

Si el servidor devuelve un código diferente (por ejemplo **503 - Service Unavailable** o **429 - Too Many Requests**), el cuerpo de la respuesta **no contiene JSON válido** — puede estar vacío o contener un mensaje de error en texto plano. Si intentamos llamar `.json()` directamente sin verificar, el programa lanzará un `JSONDecodeError`.

Por eso siempre debemos verificar que `respuesta.status_code == 200` (o equivalentemente `respuesta.ok`) **antes** de intentar parsear la respuesta.

> **Escribe en la celda de código siguiente: ¿qué imprimirías tú para que el usuario sepa qué salió mal cuando el código no es 200?**

In [15]:
URL_BASE = "https://api.nasa.gov/DONKI/GST"
startDate = "2024-05-01"
endDate   = "2024-05-31"
url = f"{URL_BASE}?startDate={startDate}&endDate={endDate}&api_key={API_KEY}"

respuesta = requests.get(url, timeout=15)

if not respuesta.ok:

    print(f"Error {respuesta.status_code}: {respuesta.text}")
    raise SystemExit("No se pudo obtener la respuesta. Intenta de nuevo.")

tormentas = respuesta.json()
print(f"Solicitud exitosa ✓ — {len(tormentas)} tormentas recibidas")

Solicitud exitosa ✓ — 5 tormentas recibidas


> ⚠️ **Cuidado con exponer tu API key**  
> Puedes descomentar el `print` de la celda siguiente para verificar que la URL se formó correctamente, pero **vuelve a comentarlo antes de guardar y entregar el notebook**. Si lo dejas activo, tu API key quedará visible en el output del archivo.

In [16]:
# Descomenta la siguiente línea para verificar que la URL se formó correctamente.
# ⚠ Vuelve a comentarla antes de entregar — el output guarda tu API key en el archivo.

# print("URL:", url)

### Parte b) — Explorar la lista de tormentas (10 pts)

Usando un ciclo `for`, imprime cuántas tormentas hubo en el período y el `gstID` y `startTime` de cada una.

In [17]:
print(f"Se encontraron un total de {len(tormentas)} tormentas geomagnéticas:")
print("-" * 50) # Línea divisoria para que se vea ordenado

for tormenta in tormentas:
    id_tormenta = tormenta["gstID"]
    fecha_inicio = tormenta["startTime"]
    
    print(f"ID: {id_tormenta} | Inicio: {fecha_inicio}")


Se encontraron un total de 5 tormentas geomagnéticas:
--------------------------------------------------
ID: 2024-05-02T15:00:00-GST-001 | Inicio: 2024-05-02T15:00Z
ID: 2024-05-10T15:00:00-GST-001 | Inicio: 2024-05-10T15:00Z
ID: 2024-05-12T21:00:00-GST-001 | Inicio: 2024-05-12T21:00Z
ID: 2024-05-16T06:00:00-GST-001 | Inicio: 2024-05-16T06:00Z
ID: 2024-05-17T18:00:00-GST-001 | Inicio: 2024-05-17T18:00Z


---
## Ejercicio 2 — Kp máximo por tormenta (20 pts)

Cada tormenta contiene una lista `allKpIndex` con múltiples mediciones a lo largo del tiempo.

**a) (10 pts)** Para cada tormenta, encuentra el **valor máximo de Kp** usando un ciclo `for` sobre `allKpIndex`. Imprime el `startTime` y el Kp máximo de cada tormenta.

**b) (10 pts)** Encuentra e imprime la tormenta **más intensa** del período (la que tuvo el mayor Kp máximo entre todas las tormentas).

In [18]:
#Parte b Inicialización de variables para el máximo global------------------------------------------
tormenta_mas_intensa = None
kp_global_max = 0

#Parte a Recorrer cada tormenta-------------------------------
for tormenta in tormentas:
   
    valores_kp = [medicion["kpIndex"] for medicion in tormenta["allKpIndex"]]
  
    kp_max = max(valores_kp)

    print(f"{tormenta['startTime']} Kp máx = {kp_max:.2f}")

    if kp_max > kp_global_max:
        kp_global_max = kp_max
        tormenta_mas_intensa = tormenta

print("-" * 50)
if tormenta_mas_intensa:
    print("LA TORMENTA MÁS INTENSA FUE:")
    print(f"Fecha: {tormenta_mas_intensa['startTime']}")
    print(f"Kp máximo global: {kp_global_max}")


2024-05-02T15:00Z Kp máx = 6.67
2024-05-10T15:00Z Kp máx = 9.00
2024-05-12T21:00Z Kp máx = 6.33
2024-05-16T06:00Z Kp máx = 6.00
2024-05-17T18:00Z Kp máx = 6.00
--------------------------------------------------
LA TORMENTA MÁS INTENSA FUE:
Fecha: 2024-05-10T15:00Z
Kp máximo global: 9.0


---
## Ejercicio 3 — Clasificar la severidad con condicionales (25 pts)

**a) (15 pts)** Escribe una función `clasificar_kp(kp)` que reciba un valor de Kp (float) y retorne una cadena con la categoría según la tabla del contexto:
- Kp < 4 → `"Quieto"`
- Kp == 4 → `"Activo"`
- Kp == 5 → `"G1 - Menor"`
- Kp == 6 → `"G2 - Moderada"`
- Kp == 7 → `"G3 - Fuerte"`
- Kp == 8 → `"G4 - Severa"`
- Kp >= 9 → `"G5 - Extrema"`

> **Nota:** el índice Kp puede ser decimal (ej. 8.67). Usa `int(kp)` para redondear hacia abajo y comparar.

**b) (10 pts)** Usa tu función para imprimir una tabla con cada tormenta, su Kp máximo y su categoría.

In [19]:
#PARTE A------------------------------------------------------------------
def clasificar_kp(kp):
    """Clasifica la intensidad de una tormenta según el índice Kp."""
    valor = int(kp)
    
    if valor < 4:
        return "Quieto"
    elif valor == 4:
        return "Activo"
    elif valor == 5:
        return "G1 - Menor"
    elif valor == 6:
        return "G2 - Moderada"
    elif valor == 7:
        return "G3 - Fuerte"
    elif valor == 8:
        return "G4 - Severa"
    else: # Esto cubre Kp >= 9
        return "G5 - Extrema"

#PARTE B-----------------------------------------------------------------
print(f"\n{'Fecha inicio':<22} {'Kp máx':>7} {'Categoría'}")
print("-" * 50)

for tormenta in tormentas:
  
    lista_kps = [m["kpIndex"] for m in tormenta["allKpIndex"]]
    
    kp_max = max(lista_kps)
    
    categoria = clasificar_kp(kp_max)
    
    fecha = tormenta['startTime']
    print(f"{fecha:<22} {kp_max:>7.2f}  {categoria}")



Fecha inicio            Kp máx Categoría
--------------------------------------------------
2024-05-02T15:00Z         6.67  G2 - Moderada
2024-05-10T15:00Z         9.00  G5 - Extrema
2024-05-12T21:00Z         6.33  G2 - Moderada
2024-05-16T06:00Z         6.00  G2 - Moderada
2024-05-17T18:00Z         6.00  G2 - Moderada


---
## Ejercicio 4 — Función de análisis completa (35 pts)

Escribe una función `analizar_tormenta(tormenta)` que reciba **un evento** de la lista `tormentas` y retorne un **diccionario** con las siguientes claves:

| Clave | Valor esperado |
|---|---|
| `"id"` | el `gstID` del evento |
| `"inicio"` | el `startTime` |
| `"kp_max"` | el valor Kp más alto entre todas las mediciones |
| `"kp_min"` | el valor Kp más bajo entre todas las mediciones |
| `"kp_promedio"` | promedio de todos los valores Kp (redondeado a 2 decimales) |
| `"num_mediciones"` | cuántas mediciones hay en `allKpIndex` |
| `"categoria"` | resultado de llamar `clasificar_kp(kp_max)` |
| `"eventos_vinculados"` | número de eventos en `linkedEvents` (0 si es `None`) |

Luego aplícala a **todas las tormentas** con un ciclo `for` e imprime un resumen de cada una.

In [20]:
def analizar_tormenta(tormenta):
   
    mediciones_kp = [m["kpIndex"] for m in tormenta.get("allKpIndex", [])]
 
    kp_max = max(mediciones_kp) if mediciones_kp else 0
    kp_min = min(mediciones_kp) if mediciones_kp else 0
    kp_promedio = round(sum(mediciones_kp) / len(mediciones_kp), 2) if mediciones_kp else 0
    
    eventos_vinculados = tormenta.get("linkedEvents")
    num_vinculados = len(eventos_vinculados) if eventos_vinculados is not None else 0

    return {
        "id": tormenta.get("gstID"),
        "inicio": tormenta.get("startTime"),
        "kp_max": kp_max,
        "kp_min": kp_min,
        "kp_promedio": kp_promedio,
        "num_mediciones": len(mediciones_kp),
        "categoria": clasificar_kp(kp_max), # Usamos tu función del ejercicio 3
        "eventos_vinculados": num_vinculados
    }


print(f"\n{'REPORTE INTEGRADO DE TORMENTAS GEOMAGNÉTICAS':^120}")
print("-" * 135)

print(f"{'ID':<18} | {'Inicio':<18} | {'Categoría':<14} | {'Mín':>6} | {'Máx':>6} | {'Prom':>6} | {'Med.':^6} | {'Vinc.'}")
print("-" * 135)

for t in tormentas:
    res = analizar_tormenta(t)
    
    print(f"{res['id']:<18} | "
          f"{res['inicio']:<18} | "
          f"{res['categoria']:<14} | "
          f"{res['kp_min']:>6.2f} | "
          f"{res['kp_max']:>6.2f} | "
          f"{res['kp_promedio']:>6.2f} | "
          f"{res['num_mediciones']:^6} | "  # Nueva columna aquí
          f"{res['eventos_vinculados']:^6}")

print("-" * 135)


                                      REPORTE INTEGRADO DE TORMENTAS GEOMAGNÉTICAS                                      
---------------------------------------------------------------------------------------------------------------------------------------
ID                 | Inicio             | Categoría      |    Mín |    Máx |   Prom |  Med.  | Vinc.
---------------------------------------------------------------------------------------------------------------------------------------
2024-05-02T15:00:00-GST-001 | 2024-05-02T15:00Z  | G2 - Moderada  |   6.67 |   6.67 |   6.67 |   2    |   2   
2024-05-10T15:00:00-GST-001 | 2024-05-10T15:00Z  | G5 - Extrema   |   6.67 |   9.00 |   8.13 |   13   |   6   
2024-05-12T21:00:00-GST-001 | 2024-05-12T21:00Z  | G2 - Moderada  |   5.67 |   6.33 |   6.00 |   3    |   2   
2024-05-16T06:00:00-GST-001 | 2024-05-16T06:00Z  | G2 - Moderada  |   6.00 |   6.00 |   6.00 |   1    |   1   
2024-05-17T18:00:00-GST-001 | 2024-05-17T18:00Z  | G2 - Moder

---
## Pregunta final — Conclusiones (incluida en el punteo del Ejercicio 4)

Basandote en los datos que obtuviste a lo largo del examen, escribe en la celda de abajo  
una conclusion en tus propias palabras. Puedes responder estas preguntas como guia:

- ¿Cual fue la tormenta mas intensa del periodo analizado y que tan severa fue segun la escala Kp?
- ¿Que patrones observas en los datos? (frecuencia, intensidad, eventos vinculados)
- ¿Que impacto podria haber tenido una tormenta de esa magnitud en la vida cotidiana?

**Conclusiones:**

pregunta 1: 2024-05-10T15:00:00-GST-001 | 2024-05-10T15:00Z  | G5 - Extrema   

pregunta 2:
Intensidad: La mayoría de los eventos registrados (4 de 5) se mantuvieron en la categoría G2 , con valores Kp máximos oscilando entre 6.00 y 6.67.

pregunta 3: 
Sistemas Eléctricos: Posibles problemas generalizados de control de voltaje.
Navegación y Comunicaciones: Interrupciones severas en la navegación por satélite (GPS) y comunicaciones de radio de alta frecuencia por varios días.
Satélites: Problemas de carga superficial y dificultades en el mantenimiento de órbita para satélites de baja altura.

---
## Sello de entrega

Ejecuta la siguiente celda **como ultimo paso**, justo antes de guardar y subir el notebook.

Realiza una consulta a la API de la ISS para registrar tu posicion geografica aproximada  
al momento de entregar. Dado que la ISS se desplaza ~30 km cada 4 segundos, dos estudiantes  
que entreguen en momentos distintos obtendran coordenadas completamente diferentes.

> Este sello es **unico e irrepetible**: queda vinculado a tu carnet y al instante  
> exacto en que ejecutaste la celda.

In [ ]:
from datetime import datetime, timezone

# Obtener la posicion actual de la ISS
print("Consultando posicion de la ISS...")
try:
    r_iss = requests.get("http://api.open-notify.org/iss-now.json", timeout=8)
    r_iss.raise_for_status()
    iss_data   = r_iss.json()
    iss_lat    = float(iss_data["iss_position"]["latitude"])
    iss_lon    = float(iss_data["iss_position"]["longitude"])
    iss_ts     = iss_data["timestamp"]
    iss_hora   = datetime.fromtimestamp(iss_ts, tz=timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")
    iss_ok     = True
except Exception as e:
    iss_lat, iss_lon, iss_hora = 0.0, 0.0, "no disponible"
    iss_ok = False
    print(f"  Advertencia: no se pudo obtener la posicion ISS ({e})")

# Timestamp local del momento de entrega
ts_entrega = datetime.now(tz=timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")

# Imprimir el sello
print()
print("=" * 55)
print("           SELLO DE ENTREGA -- PARCIAL 2")
print("=" * 55)
print(f"  Estudiante   : {NOMBRE}")
print(f"  Carnet       : {CARNET}")
print(f"  Fecha/Hora   : {ts_entrega}")
print(f"  ISS Latitud  : {iss_lat:+.4f} deg")
print(f"  ISS Longitud : {iss_lon:+.4f} deg")
print(f"  ISS Hora UTC : {iss_hora}")
print("=" * 55)

if not iss_ok:
    print("  Advertencia: sello generado sin datos de ISS (sin conexion).")
    print("  Guarda el notebook de todas formas.")

Consultando posicion de la ISS...

           SELLO DE ENTREGA -- PARCIAL 2
  Estudiante   : Rubén de Jesús Samayoa Girón
  Carnet       : 202607157
  Fecha/Hora   : 2026-04-21 20:15:04 UTC
  ISS Latitud  : +2.7350 deg
  ISS Longitud : -116.9209 deg
  ISS Hora UTC : 2026-04-21 20:15:06 UTC


---
## Entrega

1. Ejecuta la celda **Sello de entrega** (arriba)
2. Guarda el notebook: **Archivo → Guardar** (o `Ctrl+S`)
3. Verifica que **todas las celdas tienen output** (ejecuta: **Kernel → Restart & Run All**)
4. Sube el archivo a Github y copia el link en el campo de entrega en la U Virtual